# Week 7 — Chunking Strategies + Frameworks
> Goal: Compare all five chunking strategies on a real 10-K document, then build a full pipeline with LangChain and LlamaIndex.

**Topics covered:**
- Fixed / Recursive / Token / Semantic / Sentence-window chunking
- Visualizing chunk size distributions
- LangChain: `RecursiveCharacterTextSplitter`, `SemanticChunker`
- LlamaIndex: `SentenceWindowNodeParser`, `HierarchicalNodeParser`
- Qualitative retrieval comparison across strategies


## 0. Setup

In [ ]:
import sys, os, json
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from chunker import chunk_document, benchmark_strategies, Chunk
from embedder import get_embedder
from vectorstore import FAISSStore

# Sample 10-K text (use real file if available)
SAMPLE_10K = """
PART I

Item 1. Business

Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets,
wearables, and accessories worldwide. The Company also sells a variety of related services.
The Company's fiscal year is the 52 or 53-week period that ends on the last Saturday of
September. The Company's products include iPhone, Mac, iPad, Apple Watch, AirPods, Apple TV,
HomePod, and Beats products. Services include advertising, AppleCare, cloud services,
digital content, and payment services.

The Company's business strategy leverages its unique ability to design and develop its own
operating systems, hardware, application software, and services to provide its customers
products and solutions with innovative design, superior ease-of-use, and seamless integration.

Item 1A. Risk Factors

The following discussion of risk factors contains forward-looking statements.
These risk factors may be important to understanding other statements in this Form 10-K.

Global and regional economic conditions could materially adversely affect the Company's
business, results of operations, financial condition, and stock price. The Company's
operations and performance depend significantly on worldwide economic conditions.

The Company is subject to intense competition across all markets for its products and
services. The Company believes it competes primarily on innovation and quality, but many
of the Company's competitors have greater resources and could pursue strategies that would
harm the Company's competitive position.

The Company depends on third-party manufacturers. If the supply or quality of components
or manufacturing capacity is constrained or interrupted, or if prices increase, the Company's
revenue and gross margins could be adversely affected.

Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations

The following discussion should be read in conjunction with the consolidated financial
statements and accompanying notes in Part II, Item 8 of this Form 10-K.

Fiscal Year 2023 Highlights

Net sales decreased 3% or $10.9 billion to $383.3 billion during 2023 compared to 2022.
The decrease was driven by lower iPhone, Mac, iPad, and Wearables, Home and Accessories net
sales, partially offset by higher Services net sales. The weakness in foreign currencies
relative to the U.S. dollar had an unfavorable impact on net sales.

Gross margin percentage increased to 44.1% during 2023 compared to 43.3% in 2022. The
improvement was primarily driven by a favorable mix shift toward higher-margin Services revenue.
Services gross margin was 70.8% while Products gross margin was 36.6%.

Operating expenses increased 7% or $3.1 billion to $54.8 billion during 2023 compared to 2022,
driven by increases in research and development and selling, general and administrative expenses.

Item 8. Financial Statements and Supplementary Data

The following financial statements and notes are incorporated herein by reference.

CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares and per share amounts)

Net sales:
  Products                          $298,085    $316,199    $297,392
  Services                            85,200      78,129      68,425
Total net sales                      383,285     394,328     365,817

Cost of sales:
  Products                           189,282     201,471     192,266
  Services                            24,855      22,075      20,715
Total cost of sales                  214,137     223,546     212,981

Gross margin                         169,148     170,782     152,836
"""

print(f'Sample 10-K text: {len(SAMPLE_10K)} characters')

---
## 1. All Five Chunking Strategies

In [ ]:
# ── 1.1  Run all strategies ───────────────────────────────────────────────────
meta = {'company': 'AAPL', 'year': 2023, 'source': '10-K'}

strategies_to_run = ['fixed', 'recursive', 'token']
# Note: 'semantic' skipped here as it requires OpenAI API calls per split
# Run it separately if you have budget: chunk_document(text, 'semantic', meta)

chunked = {}
for strategy in strategies_to_run:
    chunks = chunk_document(SAMPLE_10K, strategy=strategy, metadata=meta)
    chunked[strategy] = chunks
    print(f'[{strategy:12s}] {len(chunks):3d} chunks | avg {sum(len(c.text) for c in chunks)//max(len(chunks),1)} chars')

In [ ]:
# ── 1.2  Visualize chunk size distributions ───────────────────────────────────
os.makedirs('plots', exist_ok=True)

fig, axes = plt.subplots(1, len(strategies_to_run), figsize=(14, 4), sharey=False)
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, (strategy, chunks), color in zip(axes, chunked.items(), colors):
    lengths = [len(c.text) for c in chunks]
    ax.hist(lengths, bins=10, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(lengths), color='red', linestyle='--', linewidth=1.5,
               label=f'mean={np.mean(lengths):.0f}')
    ax.set_title(f'{strategy}\n({len(chunks)} chunks)', fontsize=11)
    ax.set_xlabel('Chunk length (chars)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Chunk Size Distributions by Strategy', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plots/chunk_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.3  Inspect chunk boundaries ─────────────────────────────────────────────
# Does the chunk break mid-sentence or at natural boundaries?
print('=== Chunk boundary inspection ===\n')
for strategy, chunks in chunked.items():
    print(f'[{strategy}] — chunk 1/2 boundary:')
    print(f'  ...{chunks[0].text[-80:].strip()!r}')
    print(f'  {chunks[1].text[:80].strip()!r}...')
    print()

---
## 2. Sentence Window Chunking (Manual Implementation)

In [ ]:
# ── 2.1  Sentence window: embed individual sentences, store window context ─────
sw_chunks = chunk_document(SAMPLE_10K, strategy='sentence_window', metadata=meta, window_size=2)
print(f'Sentence window: {len(sw_chunks)} chunks')
print('\nExample chunk (embedded unit = first sentence, but context = ±2 window):')
for i, c in enumerate(sw_chunks[3:6], 4):
    print(f'  Chunk {i}: {c.text[:120]}...')
    print()

print('Key insight: Each chunk is a full window of 2-3 sentences.')
print('LlamaIndex SentenceWindowNodeParser does this automatically with MetadataReplacement.')

---
## 3. Retrieval Quality Comparison

In [ ]:
# ── 3.1  Build one FAISS index per strategy and compare retrieval ─────────────
embedder = get_embedder('openai-small')

test_queries = [
    {'query': 'What was Apple total revenue in 2023?',
     'expected_keywords': ['383', 'net sales', 'revenue']},
    {'query': 'What are the biggest risks for Apple?',
     'expected_keywords': ['competition', 'economic', 'supply']},
    {'query': 'What was the gross margin percentage?',
     'expected_keywords': ['44.1', 'gross margin', '43.3']},
]

comparison_results = {}

for strategy, chunks in chunked.items():
    texts = [c.text for c in chunks]
    chunk_vecs = embedder(texts)
    
    store = FAISSStore(dim=chunk_vecs.shape[1])
    store.add(chunk_vecs, [{'text': t, **c.metadata} for t, c in zip(texts, chunks)])
    
    strategy_scores = []
    for qitem in test_queries:
        q_vec = embedder([qitem['query']])[0]
        top5 = store.search(q_vec, k=5)
        combined_text = ' '.join(r['text'].lower() for r in top5)
        keyword_hits = sum(1 for kw in qitem['expected_keywords'] if kw.lower() in combined_text)
        strategy_scores.append(keyword_hits / len(qitem['expected_keywords']))
    
    avg_score = np.mean(strategy_scores)
    comparison_results[strategy] = {'per_query': strategy_scores, 'avg': avg_score}
    print(f'[{strategy:12s}] avg keyword hit rate: {avg_score:.3f}')

In [ ]:
# ── 3.2  Visualize comparison ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Per-query heatmap
strategies = list(comparison_results.keys())
n_q = len(test_queries)
data = np.array([comparison_results[s]['per_query'] for s in strategies])

im = ax1.imshow(data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax1.set_xticks(range(n_q))
ax1.set_xticklabels([f'Q{i+1}' for i in range(n_q)])
ax1.set_yticks(range(len(strategies)))
ax1.set_yticklabels(strategies)
ax1.set_title('Keyword Hit Rate per Query')
plt.colorbar(im, ax=ax1, label='Hit rate (0=miss, 1=all found)')
for i in range(len(strategies)):
    for j in range(n_q):
        ax1.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center',
                 color='black' if data[i,j] > 0.3 else 'white', fontsize=10)

# Average score bar chart  
avgs = [comparison_results[s]['avg'] for s in strategies]
bar_colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax2.barh(strategies, avgs, color=bar_colors[:len(strategies)], alpha=0.85)
ax2.set_xlim(0, 1.1)
ax2.set_xlabel('Average keyword hit rate')
ax2.set_title('Overall Retrieval Quality\n(higher = better)')
for bar, val in zip(bars, avgs):
    ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)
ax2.grid(axis='x', alpha=0.3)

q_labels = [q['query'][:30] + '…' for q in test_queries]
ax1.set_xticklabels([f'Q{i+1}: {q}' for i, q in enumerate(q_labels)],
                    rotation=20, ha='right', fontsize=8)

plt.suptitle('Chunking Strategy Comparison\n(keyword hit rate as proxy for retrieval quality)', y=1.02)
plt.tight_layout()
plt.savefig('plots/chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. LangChain Full Pipeline

In [ ]:
# ── 4.1  LangChain: build a simple RAG chain ──────────────────────────────────
from langchain.schema import Document
from langchain_community.vectorstores import FAISS as LangFAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Convert our chunks to LangChain Documents
recursive_docs = [
    Document(
        page_content=c.text,
        metadata=c.metadata
    )
    for c in chunked['recursive']
]

lc_embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = LangFAISS.from_documents(recursive_docs, lc_embeddings)

retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 12, 'lambda_mult': 0.6}
)

prompt = PromptTemplate.from_template("""
Answer the question using ONLY the provided context. Cite sources.
Context: {context}
Question: {question}
Answer:""")

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

chain = (
    {'context': retriever | (lambda docs: '\n\n'.join(d.page_content for d in docs)),
     'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke('What was Apple\'s gross margin in 2023 and what drove the change?')
print(answer)

---
## 5. LlamaIndex — Sentence Window Node Parser

In [ ]:
# ── 5.1  LlamaIndex: sentence window for precise citation ─────────────────────
from llama_index.core import Document as LIDocument, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LI_OpenAI

Settings.embed_model = OpenAIEmbedding(model='text-embedding-3-small')
Settings.llm = LI_OpenAI(model='gpt-4o-mini', temperature=0.1)

# Parse with 3-sentence window
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key='window',
    original_text_metadata_key='original_sentence',
)

li_doc = LIDocument(text=SAMPLE_10K, metadata={'company': 'AAPL', 'year': 2023})
nodes = node_parser.get_nodes_from_documents([li_doc])
print(f'Nodes created: {len(nodes)}')
print(f'\nNode 5 — original sentence:')
print(f'  {nodes[5].metadata.get("original_sentence", "")}')
print(f'\nNode 5 — window context (what gets passed to LLM):')
print(f'  {nodes[5].metadata.get("window", "")[:200]}...')

In [ ]:
# ── 5.2  Build index and query ────────────────────────────────────────────────
li_index = VectorStoreIndex(nodes)

query_engine = li_index.as_query_engine(
    similarity_top_k=3,
    node_postprocessors=[
        # Replace each matched sentence with its full window for richer context
        MetadataReplacementPostProcessor(target_metadata_key='window')
    ],
)

response = query_engine.query('What were the main factors behind Apple gross margin improvement?')
print('Answer:', response)
print('\n--- Source nodes ---')
for node in response.source_nodes:
    print(f'  [{node.score:.3f}] {node.text[:80]}...')

---
## 6. Framework Comparison

| | LangChain | LlamaIndex |
|---|---|---|
| **Strength** | Flexibility, chains, tools | Node parsers, data connectors |
| **Chunking** | `RecursiveCharacterTextSplitter`, `SemanticChunker` | `SentenceWindowNodeParser`, `HierarchicalNodeParser` |
| **Best for** | Complex chains, agents, multi-tool | Document Q&A, ingestion pipelines |
| **Learning curve** | Moderate | Moderate |

**Recommendation:** Use both. LlamaIndex for ingestion/indexing, LangChain for the query chain and tool integration.

**Next up → Week 8: Evaluating retrieval quality with RAGAS.**